In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import os
os.listdir('/content/drive/MyDrive')

['Colab Notebooks', 'MinorProject-04']

In [3]:
import os

os.chdir('/content/drive/MyDrive/MinorProject-04')
print(os.listdir())


['.virtual_documents', 'anaconda_projects', '.ipynb_checkpoints', 'samsum-train.csv', 'samsum-test.csv', 'samsum-validation.csv', 'text_summarizer.ipynb']


In [24]:
import pandas as pd
from transformers import T5Tokenizer , Trainer , TrainingArguments , T5ForConditionalGeneration

In [25]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [26]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [27]:
train_data.shape

(14732, 3)

In [28]:
val_data.shape

(818, 3)

In [29]:
# Random Sampling

train_data = train_data.sample(n = 4000 , random_state=42).reset_index(drop = True)
val_data = val_data.sample(n = 500 , random_state=42).reset_index(drop = True)

In [30]:
val_data.shape


(500, 3)

In [31]:
# Data Preprocessing:

import re

def clean_data(text):
  text = re.sub(r"\r\n" , " " , text)   # lines (next line tag)
  text = re.sub(r"\s+" , " " , text)    # spaces
  text = re.sub(r"<.*?>" , " " , text)  # html tags <p> <h1>
  # strip to delete the extra spaces
  text = text.strip().lower()
  return text

In [32]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data['dialogue'] = val_data["dialogue"].apply(clean_data)
val_data['summary'] = val_data["summary"].apply(clean_data)

In [33]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

In [34]:
# Tokenization

tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [35]:
# raw data ---> tokenized inputs for fine-tuning

def tokenize(data):
  inputs = tokenizer(data["dialogue"] , padding="max_length" , max_length=521 , truncation=True)
  targets = tokenizer(data["summary"] , padding="max_length" , max_length=150 , truncation=True)

  inputs["labels"] = targets["input_ids"]  # token ids -> add to inputs as labels  (matlb tokens ki ids jo hn woh input ka labels col bn gayi hn)
  return inputs

In [36]:
train_dataset = train_data.apply(tokenize , axis = 1).tolist()
val_dataset = val_data.apply(tokenize , axis = 1).tolist()

In [37]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [38]:
# input ids ----- dialogue => token ids  (512)

# when 1 comes , its the End of Squence EOS

# Attention mask ---- it means how many valid token ids are there{1} and when it {0} then that is not a valid id
# labels are the target ---- or the summary token (150)

# And this all in the data type of list

In [39]:
# Working with our Model
# As this is NLP project and in GenAI if we generation on some input based condition so that is Conditional Generational Task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
import torch

if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

print("Device : " , device)
model.to(device)

In [41]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay= 0.01,

    per_device_train_batch_size= 8,
    per_device_eval_batch_size= 8,

    eval_strategy= "epoch",  # after every epoch we want to evaluate or validate
    save_strategy= "epoch",  # after every epoch we want to save

    warmup_steps= 500 ,   # initially learningrate is 0 and after this steps it wil reach to its default value
    # at start it will be slow and then fast
    )

In [42]:
# Now trainer class

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [43]:
# Train The Model

trainer.train()

Epoch,Training Loss,Validation Loss
1,3.606767,0.381245
2,0.396731,0.359283
3,0.374105,0.353897
4,0.363046,0.350666
5,0.354322,0.349055
6,0.351836,0.348633


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9078011627197266, metrics={'train_runtime': 1400.7635, 'train_samples_per_second': 17.134, 'train_steps_per_second': 2.142, 'total_flos': 3305300557824000.0, 'train_loss': 0.9078011627197266, 'epoch': 6.0})

In [44]:
# model load --- fine tune ---- save the model

In [45]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [48]:
# Whenever we want to use this

model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [47]:
# Test the core logic of the summarizer


In [51]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue)   # clean the dailogue by using the clean_data func we have created above

  # tokenize
  inputs = tokenizer(
      dialogue,
      padding = "max_length",
      max_length = 512,
      truncation = True,
      return_tensors = "pt"   # it will return pytorch tensors & if we are using tensorflow so we write tf
  ).to(device)

  # generate the summary --- which will be in token ids or tokens
  model.to(device)
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length = 150,
      num_beams = 4,   # 4 diff summary sequences or generate 4 diff outputs and compare them and then give the answer (to polish output, perfomance good)
      early_stopping = True     # when 4 outputs comes then it will be the EOS(End of sequence) & we will stop
  )

  # so we convert token ids to text (summary) --decoding
  summary = tokenizer.decode(targets[0] , skip_special_tokens=True)  # EOS , seperations , tags will be skip
  return summary

In [52]:
test_dialogue = """
Ali: Hey Ahmed, have you started preparing for the upcoming exams?

Ahmed: Yes, I started last week, but I'm struggling with Machine Learning.

Ali: Which topic is difficult for you?

Ahmed: I'm having trouble understanding neural networks and backpropagation.

Ali: You should watch some tutorials and practice coding examples. That helped me a lot.

Ahmed: That's a good idea. Do you want to study together tomorrow?

Ali: Sure! Let's meet at the university library at 10 AM.

Ahmed: Perfect. I'll revise the concepts before we meet.

Ali: Great! We can solve practice questions together.

Ahmed: Thanks, Ali. See you tomorrow!
"""

summary = summarize_dialogue(test_dialogue)

print("Summary : " , summary)

Summary :  ahmed started preparing for the upcoming exams last week. ahmed is struggling with machine learning. ali will revise the concepts before meeting at the university library at 10 am.
